# 15.4 Rules for node and arc declarations

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Having defined `node` and `arc` by example, we now describe more comprehensively
the required and optional elements of these declarations, and comment on their interaction
with the conventional declarations minimize or maximize, `subject to`, and
`var` when both kinds appear in the same model.

**`node` declarations**

A `node` declaration begins with the keyword `node`, a name, an optional indexing
expression, and a colon. The expression following the colon, which describes the balance
condition at the `node`, may have any of the following forms:

```ampl
net-expr = arith-expr
net-expr <= arith-expr
net-expr >= arith-expr
arith-expr = net-expr
arith-expr <= net-expr
arith-expr >= net-expr
arith-expr <= net-expr <= arith-expr
arith-expr >= net-expr >= arith-expr
```

where an arith-expr may be any arithmetic expression that uses previously declared
model components and currently defined dummy indices. A net-expr is restricted to one
of the following:

```ampl
± net_in                                ± net_out
± net_in + arith-expr                   ± net_out + arith-expr
arith-expr ± net_in                     arith-expr ± net_out
```

(and a unary + may be omitted). Each `node` defined in this way induces a constraint in
the resulting linear program. A `node` name is treated like a constraint name in the AMPL
command environment, for example in a display statement.

For declarations that use `net_in`, AMPL generates the constraint by substituting, at
the place where `net_in` appears in the balance conditions, a linear expression that represents
flow into the `node` minus flow out of the `node`. Declarations that use `net_out` are
handled the same way, except that AMPL substitutes flow out minus flow in. The expressions
for flow in and flow out are deduced from the `arc` declarations.

**`arc` declarations**

An `arc` declaration consists of the keyword `arc`, a name, an optional indexing
expression, and a series of optional qualifying phrases. Each `arc` creates a variable in the
resulting linear program, whose value is the amount of flow over the `arc`; the `arc` name
may be used to refer to this variable elsewhere. All of the phrases that may appear in a
`var` definition have the same significance in an `arc` definition; most commonly, the >=
and <= phrases are used to specify values for lower and upper bounds on the flow along
the `arc`.
The from and to phrases specify the nodes connected by an `arc`. Usually these consist
of the keyword from or to followed by a `node` name. An `arc` is interpreted to contribute
to the flow out of the from `node`, and to the flow into the to `node`; these interpretations
are what permit the inference of the constraints associated with the nodes.

Typically one from and one to phrase are specified in an `arc` declaration. Either
may be omitted, however, as in Figure 15-12. Either may also be followed by an optional
indexing expression, which should be one of two kinds:

- An indexing expression that specifies — depending on the data — an empty set (in
which case the from or to phrase is ignored) or a set with one member (in which
case the from or to phrase is used).
- An indexing expression of the special form {if logical-expr}, which causes the
from or to phrase to be used if and only if the logical-expr evaluates to true.

It is possible to specify that an `arc` carries flow out of or into two or more nodes, by giving
more than one from or to phrase, or by using an indexing expression that specifies a
set having more than one member. The result is not a network linear program, however,
and AMPL displays an appropriate warning message.

At the end of a from or to phrase, you may add an arithmetic expression representing
a factor to multiply the flow, as shown in our examples of shipping-loss and changeof-unit
variations in [Section 15.3](./15_3_declaring_network_models_by_node_and_arc.ipynb). If the factor is in the to phrase, it multiplies the `arc`
variable in determining the flow into the specified `node`; that is, for a given flow along the
`arc`, an amount equal to the to-factor times the flow is considered to enter the to `node`.
A factor in the from phrase is interpreted analogously. The default factor is 1.

An optional obj phrase specifies a coefficient that will multiply the `arc` variable to
create a linear term in a specified objective function. Such a phrase consists of the keyword
obj, the name of an objective that has previously been defined in a minimize or
maximize declaration, and an arithmetic expression for the coefficient value. The keyword
may be followed by an indexing expression, which is interpreted as for the from
and to phrases.

**Interaction with objective declarations**

If all terms in the objective function are specified through obj phrases in `arc` declarations,
the declaration of the objective is simply minimize or maximize followed by
an optional indexing expression and a name. This declaration must come before the `arc`
declarations that refer to the objective.

Alternatively, `arc` names may be used as variables to specify the objective function in
the usual algebraic way. In this case the objective must be declared after the arcs, as in
Figure 15-12.

```ampl
set CITIES;
set LINKS within (CITIES cross CITIES);
set PRODS;
param supply {CITIES,PRODS} >= 0;  # amounts available at cities
param demand {CITIES,PRODS} >= 0;  # amounts required at cities
       check {p in PRODS}:
              sum {i in CITIES} supply[i,p] = sum {j in CITIES} demand[j,p];
param cost {LINKS,PRODS} >= 0;     # shipment costs/1000 packages
param capacity {LINKS,PRODS} >= 0; # max packages shipped
param cap_joint {LINKS} >= 0;      # max total packages shipped/link
minimize Total_Cost;
node Balance {k in CITIES, p in PRODS}:
net_in = demand[k,p] - supply[k,p];
arc Ship {(i,j) in LINKS, p in PRODS} >= 0, <= capacity[i,j,p],
from Balance[i,p], to Balance[j,p], obj Total_Cost cost[i,j,p];
subject to Multi {(i,j) in LINKS}:
sum {p in PRODS} Ship[i,j,p] <= cap_joint[i,j];
```

**Figure 15-13:** Multicommodity flow with side constraints (netmulti.mod).

Interaction with constraint declarations
The components defined in `arc` declarations may be used as variables in additional
`subject to` declarations. The latter represent "side constraints" that are imposed in
addition to balance of flow at the nodes.

As an example, consider how a multicommodity flow problem can be built from the
`node`-and-`arc` network formulation in [Figure 15-10](./15_3_declaring_network_models_by_node_and_arc.ipynb#fig-15-10). Following the approach in Section 4.1,
we introduce a set PRODS of different products, and add it to the indexing of all
parameters, nodes and arcs. The result is a separate network linear program for each
product, with the objective function being the sum of the costs for all products. To tie
these networks together, we provide for a joint limit on the total shipments along any
link:

```ampl
param cap_joint {LINKS} >= 0;
subject to Multi {(i,j) in LINKS}:
       sum {p in PRODS} Ship[p,i,j] <= cap_joint[i,j];
```

The final model, shown in Figure 15-13, is not a network linear program, but the network
and non-network parts of it are cleanly separated.

Interaction with variable declarations
Just as an `arc` variable may be used in a `subject to` declaration, an ordinary `var`
variable may be used in a `node` declaration. That is, the balance condition in a `node`
declaration may contain references to variables that were defined by preceding `var` declarations.
These references define "side variables" to the network linear program.
As an example, we again replicate the formulation of [Figure 15-10](./15_3_declaring_network_models_by_node_and_arc.ipynb#fig-15-10) over the set
PRODS. This time we tie the networks together by introducing a set of feedstocks and
associated data:

```ampl
set FEEDS;
param yield {PRODS,FEEDS} >= 0;
param limit {FEEDS,CITIES} >= 0;
```

We imagine that at city k, in addition to the amounts supply[p,k] of products available
to be shipped, up to limit[f,k] of feedstock f can be converted into products;
one unit of feedstock f gives rise to yield[p,f] units of each product p. A variable
Feed[f,k] represents the amount of feedstock f used at city k:

```ampl
var Feed {f in FEEDS, k in CITIES} >= 0, <= limit[f,k];
```

The balance condition for product `p` at city k can now say that the net flow out equals net
supply plus the sum of the amounts derived from the various feedstocks:

```ampl
node Balance {p in PRODS, k in CITIES}:
       net_out = supply[p,k] - demand[p,k]
       + sum {f in FEEDS} yield[p,f] * Feed[f,k];
```

The arcs are unchanged, leading to the model shown in Figure 15-14. At a given city k,
the variables Feed[f,k] appear in the `node` balance conditions for all the different
products, bringing together the product networks into a single linear program.